# Heart Disease Classification - Model Training

This notebook implements a complete **MLOps training pipeline** for heart disease prediction, including:
- Training multiple classification models (Logistic Regression, Random Forest, Gradient Boosting, SVM)
- Hyperparameter tuning with GridSearchCV
- Cross-validation for robust performance estimation
- MLflow experiment tracking for reproducibility
- Model comparison and selection

## Pipeline Overview
```
Data Loading → Preprocessing → Model Training → Evaluation → Model Registry
                                    ↓
                              MLflow Tracking
```

**Key MLOps Practices Demonstrated:**
1. Experiment tracking with MLflow
2. Hyperparameter logging
3. Model versioning
4. Metric comparison across runs

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys

# ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
import joblib

# MLflow
import mlflow
import mlflow.sklearn

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Add project root to path
sys.path.insert(0, '..')

print("Libraries imported successfully!")

## 1. Load Preprocessed Data

We use the `HeartDiseasePreprocessor` class which provides:
- **Consistent preprocessing**: Same transformations applied to train/test data
- **Pipeline persistence**: Save fitted preprocessor for deployment
- **Feature scaling**: StandardScaler for numeric features

The 80/20 train-test split uses **stratification** to preserve class distribution.

In [ ]:
from src.data.preprocessing import (
    load_raw_data, prepare_data, handle_missing_values,
    HeartDiseasePreprocessor, get_project_root
)

# Load and prepare data
project_root = get_project_root()
df = load_raw_data()
df = handle_missing_values(df)

# Prepare train/test split
X_train, X_test, y_train, y_test, preprocessor = prepare_data(
    df, test_size=0.2, random_state=42, scale=True
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeatures: {list(X_train.columns)}")

## 2. MLflow Setup

**MLflow** is an open-source platform for managing the ML lifecycle:
- **Tracking**: Log parameters, metrics, and artifacts for each experiment
- **Projects**: Package ML code in a reproducible format
- **Models**: Deploy models from any ML library
- **Registry**: Store and manage models centrally

We use **SQLite backend** for local tracking (production would use a remote server).

```
View MLflow UI: mlflow ui --host 0.0.0.0 --port 5000
```

In [ ]:
# Configure MLflow with SQLite backend
mlflow.set_tracking_uri(f"sqlite:///{project_root / 'mlruns.db'}")
mlflow.set_experiment("heart-disease-classification")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: heart-disease-classification")

## 3. Model Definitions

We train four different algorithms to compare their performance:

| Model | Description | Strengths |
|-------|-------------|-----------|
| **Logistic Regression** | Linear classifier with regularization | Interpretable, fast, baseline |
| **Random Forest** | Ensemble of decision trees | Robust to outliers, feature importance |
| **Gradient Boosting** | Sequential ensemble learning | High accuracy, handles non-linearity |
| **SVM** | Kernel-based classifier | Works well in high dimensions |

Each model has a **hyperparameter grid** for tuning via GridSearchCV.

In [ ]:
# Define models with hyperparameter grids
MODELS = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5]
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear']
        }
    }
}

print(f"Models to train: {list(MODELS.keys())}")

## 4. Helper Functions

These utility functions support the training pipeline:

- **`evaluate_model()`**: Calculate classification metrics (accuracy, precision, recall, F1, ROC-AUC)
- **`train_with_mlflow()`**: Train model with hyperparameter search and log to MLflow
- **`plot_confusion_matrix()`**: Visualize prediction results
- **`plot_roc_curve()`**: Plot ROC curves for model comparison

**Key Metrics Explained:**
- **Accuracy**: Overall correctness (TP+TN)/(Total)
- **Precision**: Of predicted positives, how many are correct (TP/(TP+FP))
- **Recall**: Of actual positives, how many were found (TP/(TP+FN))
- **F1 Score**: Harmonic mean of precision and recall
- **ROC-AUC**: Area under the ROC curve (0.5=random, 1.0=perfect)

In [ ]:
def evaluate_model(model, X_test, y_test):
    """Evaluate model and return metrics."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    
    return metrics, y_pred, y_prob


def plot_confusion_matrix(y_true, y_pred, title):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix - {title}')
    ax.set_xticklabels(['No Disease', 'Disease'])
    ax.set_yticklabels(['No Disease', 'Disease'])
    
    return fig


def plot_roc_curve(y_true, y_prob, title, auc_score):
    """Plot ROC curve."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color='blue', lw=2, label=f'ROC (AUC = {auc_score:.3f})')
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {title}')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    return fig

## 5. Train and Evaluate Models

This is the main training loop that:

1. **For each model:**
   - Starts an MLflow run for tracking
   - Performs **5-fold cross-validated GridSearchCV** for hyperparameter tuning
   - Evaluates best model on held-out test set
   - Logs all metrics, parameters, and artifacts to MLflow

2. **Cross-validation** prevents overfitting by training on multiple data subsets

3. **Scoring metric: ROC-AUC** - balances sensitivity and specificity, important for medical diagnosis

In [ ]:
# Store results
results = {}

for model_name, config in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    with mlflow.start_run(run_name=model_name):
        # Log parameters
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_train_samples", len(X_train))
        mlflow.log_param("n_test_samples", len(X_test))
        
        # Hyperparameter tuning
        print("Performing hyperparameter tuning...")
        grid_search = GridSearchCV(
            config['model'],
            config['params'],
            cv=5,
            scoring='roc_auc',
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        
        print(f"Best parameters: {best_params}")
        print(f"Best CV score: {grid_search.best_score_:.4f}")
        
        # Log best parameters
        for param, value in best_params.items():
            mlflow.log_param(f"best_{param}", value)
        
        # Evaluate on test set
        metrics, y_pred, y_prob = evaluate_model(best_model, X_test, y_test)
        
        print(f"\nTest Metrics:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")
            mlflow.log_metric(f"test_{metric}", value)
        
        # Log CV score
        mlflow.log_metric("cv_roc_auc", grid_search.best_score_)
        
        # Save plots
        cm_fig = plot_confusion_matrix(y_test, y_pred, model_name)
        cm_path = f"../screenshots/{model_name.replace(' ', '_').lower()}_confusion_matrix.png"
        cm_fig.savefig(cm_path, dpi=150, bbox_inches='tight')
        mlflow.log_artifact(cm_path)
        plt.close(cm_fig)
        
        roc_fig = plot_roc_curve(y_test, y_prob, model_name, metrics['roc_auc'])
        roc_path = f"../screenshots/{model_name.replace(' ', '_').lower()}_roc_curve.png"
        roc_fig.savefig(roc_path, dpi=150, bbox_inches='tight')
        mlflow.log_artifact(roc_path)
        plt.close(roc_fig)
        
        # Log model
        mlflow.sklearn.log_model(best_model, "model")
        
        # Store results
        results[model_name] = {
            'model': best_model,
            'metrics': metrics,
            'best_params': best_params,
            'cv_score': grid_search.best_score_
        }

print("\n" + "="*60)
print("All models trained successfully!")
print("="*60)

## 6. Model Comparison

After training all models, we compare their performance across multiple metrics:
- **Test set metrics**: Final evaluation on held-out data
- **Cross-validation scores**: Estimate of generalization performance

The comparison DataFrame is sorted by **ROC-AUC**, our primary metric for medical classification tasks where both false positives and false negatives have significant costs.

In [ ]:
# Create comparison DataFrame
comparison_data = []
for model_name, result in results.items():
    row = {'Model': model_name}
    row.update(result['metrics'])
    row['CV Score'] = result['cv_score']
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')
comparison_df = comparison_df.sort_values('roc_auc', ascending=False)

print("Model Comparison:")
print("="*80)
comparison_df.round(4)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for metrics
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
comparison_df[metrics_to_plot].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Model Comparison - All Metrics', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

# ROC-AUC comparison
colors = plt.cm.viridis(np.linspace(0, 0.8, len(comparison_df)))
comparison_df['roc_auc'].plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('ROC-AUC Comparison', fontsize=12, fontweight='bold')
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_xlim(0.5, 1)

for i, v in enumerate(comparison_df['roc_auc']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Select and Save Best Model

The model with highest ROC-AUC is selected for deployment. We save:

1. **Model file** (`final_model.joblib`): Contains trained model + metadata
2. **Preprocessor** (`preprocessor.joblib`): Fitted StandardScaler for inference

**Saved metadata includes:**
- Model name and type
- Version number for tracking
- Training timestamp
- Performance metrics

This enables **reproducibility** and **model lineage** tracking.

In [ ]:
# Select best model based on ROC-AUC
best_model_name = comparison_df['roc_auc'].idxmax()
best_model = results[best_model_name]['model']
best_metrics = results[best_model_name]['metrics']

print(f"Best Model: {best_model_name}")
print(f"\nMetrics:")
for metric, value in best_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Save best model
from datetime import datetime

models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Save model
model_data = {
    'model': best_model,
    'model_name': best_model_name.replace(' ', '_').lower(),
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(),
    'metrics': best_metrics
}

model_path = models_dir / 'final_model.joblib'
joblib.dump(model_data, model_path)
print(f"Model saved to: {model_path}")

# Save preprocessor
preprocessor_path = models_dir / 'preprocessor.joblib'
preprocessor.save(preprocessor_path)
print(f"Preprocessor saved to: {preprocessor_path}")

## 8. Feature Importance Analysis

**Feature importance** from ensemble models (Random Forest, Gradient Boosting) shows which features most influence predictions:

- **Mean Decrease Impurity (MDI)**: Measures how much each feature reduces uncertainty
- Higher importance = more influential for prediction
- Helps identify:
  - Key clinical indicators for diagnosis
  - Potential features to collect in future studies
  - Candidates for feature engineering

Note: Importance from tree-based models may be biased toward high-cardinality features.

In [ ]:
# Get feature importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance (Random Forest):")
print("="*40)
print(feature_importance.to_string(index=False))

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.viridis(np.linspace(0, 0.8, len(feature_importance)))
feature_importance.plot(kind='barh', x='feature', y='importance', ax=ax, 
                        legend=False, color=colors)
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../screenshots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Classification Report & Summary

The **classification report** provides per-class metrics:
- **Precision**: Exactness - low false positive rate
- **Recall**: Completeness - low false negative rate  
- **F1-score**: Balance between precision and recall
- **Support**: Number of samples per class

**For medical diagnosis:**
- High **recall** is often preferred (minimize missed cases)
- The trade-off between precision and recall depends on clinical context

### Next Steps
1. Deploy model via FastAPI (see `src/api/`)
2. Monitor predictions in production
3. Retrain periodically with new data

In [ ]:
# Generate final classification report for best model
y_pred_final = best_model.predict(X_test)

print(f"Classification Report - {best_model_name}")
print("="*60)
print(classification_report(y_test, y_pred_final, target_names=['No Disease', 'Disease']))

In [ ]:
# Summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nBest Model: {best_model_name}")
print(f"\nTest Set Performance:")
print(f"  - Accuracy:  {best_metrics['accuracy']:.4f}")
print(f"  - Precision: {best_metrics['precision']:.4f}")
print(f"  - Recall:    {best_metrics['recall']:.4f}")
print(f"  - F1 Score:  {best_metrics['f1_score']:.4f}")
print(f"  - ROC-AUC:   {best_metrics['roc_auc']:.4f}")
print(f"\nModel artifacts saved to: {models_dir}")
print(f"MLflow experiments saved to: {project_root / 'mlruns'}")
print(f"\nTo view MLflow UI, run: mlflow ui --port 5000")